# Numerai Dashboard

A live view of your Numerai models. Compare finalized CORR60 and MMC60 scores, explore their percentiles, and see how performance evolves as rounds resolve.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re

import altair as alt
import mercury as mr
import pandas as pd
import requests
from IPython.display import Markdown, display

API_URL = 'https://api-tournament.numer.ai'
CACHE_DIR = Path.home() / '.cache' / 'numerai-conference-dashboard'
SNAPSHOT_DIR = Path('numerai-dashboard-data')
CACHE_MINUTES = 60
LAST_ROUNDS = 120
CACHE_DIR.mkdir(parents=True, exist_ok=True)
alt.data_transformers.disable_max_rows()

PROFILE_QUERY = '''
query($name: String!) {
  v3UserProfile(modelName: $name) { id username tournament }
}
'''
ROUNDS_QUERY = '''
query($modelId: String!) {
  v2RoundModelPerformances(modelId: $modelId, tournament: 8) {
    roundNumber
    roundResolved
    submissionScores { displayName value percentile day }
  }
}
'''

In [ ]:
def graphql(query, variables):
    response = requests.post(API_URL, json={'query': query, 'variables': variables}, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if payload.get('errors'):
        raise ValueError(payload['errors'])
    return payload['data']

def fetch_model(name):
    profile = graphql(PROFILE_QUERY, {'name': name})['v3UserProfile']
    if not profile or profile['tournament'] != 8:
        raise ValueError(f'No Numerai Classic model named {name!r}')
    rounds = graphql(ROUNDS_QUERY, {'modelId': profile['id']})['v2RoundModelPerformances']
    return {'name': profile['username'], 'rounds': rounds or []}

def has_resolution_flags(history):
    return all('roundResolved' in item for item in history.get('rounds', []))

def load_model(name):
    if not re.fullmatch(r'[a-zA-Z0-9_-]+', name):
        raise ValueError(f'Invalid model name: {name!r}')
    cache_file = CACHE_DIR / f'{name}.json'
    snapshot_file = SNAPSHOT_DIR / f'{name}.json'
    if cache_file.exists():
        age_minutes = (datetime.now(timezone.utc).timestamp() - cache_file.stat().st_mtime) / 60
        cached = json.loads(cache_file.read_text())
        if age_minutes < CACHE_MINUTES and has_resolution_flags(cached):
            return cached, 'cache'
    try:
        history = fetch_model(name)
        cache_file.write_text(json.dumps(history))
        return history, 'live GraphQL'
    except (requests.RequestException, ValueError, KeyError) as error:
        for path, source in [(cache_file, 'stale cache'), (snapshot_file, 'bundled snapshot')]:
            if path.exists():
                fallback = json.loads(path.read_text())
                if has_resolution_flags(fallback):
                    return fallback, source
        raise

In [ ]:
model_names = mr.TextInput(
    label='Numerai models (comma-separated)',
    value='v53_lgbm_ender60, integration_test, v53_lgbm_ender20',
)

In [ ]:
names = list(dict.fromkeys(
    name.strip().lower() for name in model_names.value.split(',') if name.strip()
))
histories, sources, errors = {}, {}, []
for name in names:
    try:
        histories[name], sources[name] = load_model(name)
    except (requests.RequestException, ValueError, KeyError) as error:
        errors.append(f'{name}: {error}')

rows = []
for model_name, history in histories.items():
    for round_data in history['rounds']:
        if round_data['roundResolved'] is not True:
            continue
        for score in round_data.get('submissionScores') or []:
            if (score['displayName'] not in {'corr60', 'mmc60'}
                    or score['value'] is None or (score['day'] or 0) < 60):
                continue
            rows.append({
                'model': model_name, 'round': round_data['roundNumber'],
                'metric': score['displayName'], 'score': score['value'],
                'percentile': None if score['percentile'] is None else 100 * score['percentile'],
                'day': score['day'],
            })
scores = pd.DataFrame(rows, columns=['model', 'round', 'metric', 'score', 'percentile', 'day'])
if not scores.empty:
    scores = scores[scores['round'] >= scores['round'].max() - LAST_ROUNDS + 1].copy()

In [ ]:
plot_type = mr.Select(
    label='Plot type',
    choices=['CORR60', 'MMC60', 'CORR60 percentile', 'MMC60 percentile'],
    value='CORR60',
)

In [ ]:
metric = 'corr60' if plot_type.value.startswith('CORR60') else 'mmc60'
percentile_view = plot_type.value.endswith('percentile')
series = scores[scores['metric'] == metric].copy()
series['plot_value'] = series['percentile'] if percentile_view else series['score']
series = series.dropna(subset=['plot_value'])

latest_round = str(int(scores['round'].max())) if not scores.empty else 'n/a'
if series.empty:
    max_value, max_source = 'n/a', None
else:
    best = series.loc[series['plot_value'].idxmax()]
    max_value = f"{best['plot_value']:.1f}%" if percentile_view else f"{best['plot_value']:+.4f}"
    max_source = f"{best['model']} | round {int(best['round'])}"

mr.Indicator([
    mr.Indicator(value=str(len(histories)), label='Models'),
    mr.Indicator(value=str(scores['round'].nunique()), label='Resolved rounds'),
    mr.Indicator(value=latest_round, label='Latest resolved round'),
    mr.Indicator(value=max_value, label=f'Max {plot_type.value}', delta=max_source),
])

In [ ]:
if series.empty:
    display(Markdown('No scores for these models. Try another public model name.'))
else:
    y_scale = alt.Scale(domain=[0, 100]) if percentile_view else alt.Scale(zero=True)
    base = alt.Chart(series).encode(
        x=alt.X('round:Q', title='Numerai round', scale=alt.Scale(zero=False)),
        y=alt.Y('plot_value:Q', title=plot_type.value, scale=y_scale),
        color=alt.Color('model:N', title='Model', scale=alt.Scale(scheme='tableau10')),
        tooltip=[
            alt.Tooltip('model:N'), alt.Tooltip('round:Q'),
            alt.Tooltip('score:Q', format='.4f'),
            alt.Tooltip('percentile:Q', title='Percentile (%)', format='.1f'),
            alt.Tooltip('day:Q'),
        ],
    )
    line = base.mark_line(point=True, strokeWidth=2)
    display(line.properties(
        title=f'{plot_type.value} | last {LAST_ROUNDS} rounds', width='container', height=420
    ).interactive())

if errors:
    display(Markdown('**Skipped models:** ' + '; '.join(errors)))
display(Markdown('**Data sources:** ' + ', '.join(f'{name}: {source}' for name, source in sources.items())))